# 📊 Model Evaluation and Comprehensive Reporting
## Cervical Cancer Risk Prediction

**Author:** ML Pipeline  
**Date:** 2026-07-10  
**Purpose:** Final evaluation of all models with visualizations and recommendations

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'processed_data'
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUT_DIR = PROJECT_ROOT / 'evaluation_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print("Libraries imported successfully")
print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Load Data and Models

In [ ]:
print("=" * 70)
print("LOADING DATA AND LEAKAGE-SAFE MODELS")
print("=" * 70)

X_train = np.load(DATA_DIR / 'X_train.npy')
X_val = np.load(DATA_DIR / 'X_val.npy')
X_test = np.load(DATA_DIR / 'X_test.npy')
y_train = np.load(DATA_DIR / 'y_train.npy')
y_val = np.load(DATA_DIR / 'y_val.npy')
y_test = np.load(DATA_DIR / 'y_test.npy')

with open(DATA_DIR / 'feature_columns.json', 'r') as f:
    feature_names = json.load(f)

print(f"Data loaded: train={X_train.shape}, validation={X_val.shape}, test={X_test.shape}")

# Only load artifacts produced by the leakage-safe training notebook.
model_names = [
    'Logistic_Regression',
    'Logistic_ElasticNet',
    'Linear_SVM',
    'RBF_SVM',
    'Random_Forest',
    'Gradient_Boosting',
    'LDA_Shrinkage',
    'SGD_ElasticNet',
    'Hist_Gradient_Boosting',
]
models = {}
for model_name in model_names:
    model_file = MODELS_DIR / f'{model_name}_tuned.pkl'
    if model_file.exists():
        models[model_name] = joblib.load(model_file)
        print(f"Loaded pipeline: {model_name}")
    else:
        print(f"Not found: {model_file}")

if not models:
    raise FileNotFoundError("No leakage-safe trained pipelines were found in models/.")
print(f"Total models loaded: {len(models)}")

## 3. Evaluate All Models on All Datasets

In [ ]:
print("=" * 70)
print("EVALUATING LEAKAGE-SAFE MODELS")
print("=" * 70)

all_results = {}

for model_name, model in models.items():
    print(f"\nEvaluating: {model_name}")
    results = {}
    for split_name, features, y_true in [
        ('Train', X_train, y_train),
        ('Validation', X_val, y_val),
        ('Test', X_test, y_test),
    ]:
        y_pred = model.predict(features)
        y_prob = model.predict_proba(features)[:, 1]
        results[split_name] = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1_score': f1_score(y_true, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_true, y_prob),
            'pr_auc': average_precision_score(y_true, y_prob),
            'confusion_matrix': confusion_matrix(y_true, y_pred),
            'classification_report': classification_report(
                y_true, y_pred, output_dict=True, zero_division=0
            ),
        }

    all_results[model_name] = results
    print(
        f"  Test F1: {results['Test']['f1_score']:.4f} | "
        f"Test PR-AUC: {results['Test']['pr_auc']:.4f}"
    )

print("\n" + "=" * 70)
print("ALL MODELS EVALUATED SUCCESSFULLY")
print("=" * 70)

## 4. Model Comparison Table

In [ ]:
print("\n" + "=" * 70)
print("COMPREHENSIVE MODEL COMPARISON")
print("=" * 70)

comparison_data = []
for model_name, results in all_results.items():
    row = {'Model': model_name}
    for split in ['Train', 'Validation', 'Test']:
        for metric in ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'pr_auc']:
            row[f'{split}_{metric}'] = results[split][metric]
    row['Train_Validation_Gap'] = (
        results['Train']['f1_score'] - results['Validation']['f1_score']
    )
    comparison_data.append(row)

df_comparison = pd.DataFrame(comparison_data)
df_comparison = df_comparison.sort_values(
    'Validation_f1_score', ascending=False
).reset_index(drop=True)

print("\nF1-Score Comparison:")
display_cols = [
    'Model', 'Train_f1_score', 'Validation_f1_score', 'Test_f1_score',
    'Train_Validation_Gap', 'Validation_pr_auc', 'Test_pr_auc'
]
print(df_comparison[display_cols].to_string(index=False))

print("\nROC-AUC Comparison:")
display_cols_auc = ['Model', 'Train_roc_auc', 'Validation_roc_auc', 'Test_roc_auc']
print(df_comparison[display_cols_auc].to_string(index=False))

df_comparison.to_csv(OUTPUT_DIR / 'comprehensive_comparison.csv', index=False)
print(f"\nSaved: {OUTPUT_DIR / 'comprehensive_comparison.csv'}")

## 5. Visualizations

### 5.1 F1-Score Comparison Across Datasets

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(df_comparison))
width = 0.25

train_f1 = df_comparison['Train_f1_score'].values
val_f1 = df_comparison['Validation_f1_score'].values
test_f1 = df_comparison['Test_f1_score'].values

ax.bar(x - width, train_f1, width, label='Train', alpha=0.8)
ax.bar(x, val_f1, width, label='Validation', alpha=0.8)
ax.bar(x + width, test_f1, width, label='Test', alpha=0.8)

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('F1-Score Comparison: Train vs Validation vs Test', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_comparison['Model'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_f1_score_comparison.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {OUTPUT_DIR / '01_f1_score_comparison.png'}")
plt.close()

### 5.2 Overfitting Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

gaps = df_comparison['Train_Validation_Gap'].values
colors = ['#e74c3c' if gap > 0.05 else '#2ecc71' for gap in gaps]
bars = ax.barh(df_comparison['Model'], gaps, color=colors, alpha=0.8)
ax.axvline(x=0.05, color='orange', linestyle='--', linewidth=2, label='Reference gap (0.05)')
ax.set_xlabel('Train-Validation F1 Gap', fontsize=12, fontweight='bold')
ax.set_ylabel('Model', fontsize=12, fontweight='bold')
ax.set_title('Overfitting Analysis: Train vs Validation F1 Gap', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

for bar, gap in zip(bars, gaps):
    ax.text(gap + 0.005, bar.get_y() + bar.get_height() / 2, f'{gap:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_overfitting_analysis.png', dpi=300, bbox_inches='tight')
print(f"Saved: {OUTPUT_DIR / '02_overfitting_analysis.png'}")
plt.close()

### 5.3 ROC-AUC Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(df_comparison))
width = 0.25

train_auc = df_comparison['Train_roc_auc'].values
val_auc = df_comparison['Validation_roc_auc'].values
test_auc = df_comparison['Test_roc_auc'].values

ax.bar(x - width, train_auc, width, label='Train', alpha=0.8)
ax.bar(x, val_auc, width, label='Validation', alpha=0.8)
ax.bar(x + width, test_auc, width, label='Test', alpha=0.8)

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('ROC-AUC', fontsize=12, fontweight='bold')
ax.set_title('ROC-AUC Comparison: Train vs Validation vs Test', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_comparison['Model'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0.5, 1.0])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_roc_auc_comparison.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {OUTPUT_DIR / '03_roc_auc_comparison.png'}")
plt.close()

### 5.4 Confusion Matrices for Best Model

In [ ]:
# Get best model
best_model_name = df_comparison.iloc[0]['Model']
print(f"\n🏆 Best Model: {best_model_name}")
print(f"Visualizing confusion matrices for all three sets...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

splits = ['Train', 'Validation', 'Test']
for idx, split in enumerate(splits):
    cm = all_results[best_model_name][split]['confusion_matrix']
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'],
                ax=axes[idx], cbar=False)
    
    axes[idx].set_title(f'{split} Set', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=10)
    axes[idx].set_ylabel('Actual', fontsize=10)

plt.suptitle(f'Confusion Matrices - {best_model_name}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_confusion_matrices_best_model.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {OUTPUT_DIR / '04_confusion_matrices_best_model.png'}")
plt.close()

## 6. Detailed Performance Report

In [ ]:
print("\n" + "=" * 70)
print("DETAILED PERFORMANCE REPORT")
print("=" * 70)

best_model_name = df_comparison.iloc[0]['Model']
best_results = all_results[best_model_name]

print(f"\nBest Model: {best_model_name}")
print("\nPERFORMANCE METRICS")

for split in ['Train', 'Validation', 'Test']:
    print(f"\n{split} Set:")
    print(f"  Accuracy:  {best_results[split]['accuracy']:.4f}")
    print(f"  Precision: {best_results[split]['precision']:.4f}")
    print(f"  Recall:    {best_results[split]['recall']:.4f}")
    print(f"  F1-Score:  {best_results[split]['f1_score']:.4f}")
    print(f"  ROC-AUC:   {best_results[split]['roc_auc']:.4f}")
    print(f"  PR-AUC:    {best_results[split]['pr_auc']:.4f}")
    cm = best_results[split]['confusion_matrix']
    print(f"  Confusion Matrix: TN={cm[0][0]}, FP={cm[0][1]}, FN={cm[1][0]}, TP={cm[1][1]}")

print("\nOVERFITTING ANALYSIS")
for _, row in df_comparison.iterrows():
    gap = row['Train_Validation_Gap']
    status = "REFERENCE GAP EXCEEDED" if gap > 0.05 else "WITHIN REFERENCE GAP"
    print(f"{row['Model']:25s}: {status} ({gap:.4f})")

## 7. Final Recommendations

In [ ]:
print("\n" + "=" * 70)
print("FINAL RECOMMENDATIONS")
print("=" * 70)

print("\nTOP 3 MODELS (ranked by validation F1):")
for rank, (_, row) in enumerate(df_comparison.head(3).iterrows(), 1):
    gap = row['Train_Validation_Gap']
    recommendation = "RECOMMENDED" if gap <= 0.05 else "USE WITH CAUTION"
    print(f"\n{rank}. {row['Model']}")
    print(f"   Validation F1: {row['Validation_f1_score']:.4f}")
    print(f"   Validation PR-AUC: {row['Validation_pr_auc']:.4f}")
    print(f"   Test F1: {row['Test_f1_score']:.4f}")
    print(f"   Train-validation gap: {gap:.4f} ({recommendation})")

print(f"\nBEST MODEL: {best_model_name}")
print(f"Selected by validation F1, with test metrics reported only for final assessment.")
print(f"\nOutput files are stored in {OUTPUT_DIR}/")